# Chapter 7 — Instruction Fine-tuning

> Sách: **Build a Large Language Model (From Scratch)** — Sebastian Raschka  
> Mục tiêu: Hiểu và thực hành fine-tuning một GPT đã pretrain để **làm theo hướng dẫn** và sinh câu trả lời.

## 1. Bức tranh tổng quan

- **Chapter 5 — Pretraining:** Dự đoán token tiếp theo trên văn bản.
- **Chapter 6 — Classification fine-tuning:** Dự đoán nhãn (ví dụ spam/ham).
- **Chapter 7 — Instruction fine-tuning:** Dự đoán các token trong những ví dụ *instruction → response*, để học cách trả lời yêu cầu.

```text
Pretrained GPT
     |
Instruction dataset (instruction, input, output)
     |
Prompt formatting → Tokenization → Collate & padding
     |
Token-level cross-entropy → Backpropagation → AdamW
     |
Instruction-fine-tuned GPT → Generate response → Evaluation
```

**Lưu ý:** Chapter 6 và Chapter 7 là hai nhánh fine-tuning khác nhau. Không cần fine-tune mô hình phân loại rồi mới làm instruction fine-tuning.

## 2. Instruction fine-tuning là gì?

GPT pretrained học quy luật ngôn ngữ bằng next-token prediction. Instruction fine-tuning tiếp tục huấn luyện GPT trên các cặp **yêu cầu và câu trả lời mẫu** để mô hình học cách làm theo hướng dẫn.

```text
Instruction: Explain machine learning simply.
Response: Machine learning helps computers learn patterns from data.
```

Điểm cốt lõi: **Vẫn là next-token prediction**, nhưng dữ liệu được tổ chức để dạy hành vi trả lời yêu cầu.

## 3. Phân biệt Chapter 6 và Chapter 7

| Nội dung | Chapter 6 | Chapter 7 |
|---|---|---|
| Nhiệm vụ | Phân loại văn bản | Sinh câu trả lời theo instruction |
| Target | Một class ID | Một chuỗi token |
| Output head | `Linear(768, 2)` (GPT-2 Small, 2 class) | Giữ LM head `Linear(1024, 50257)` (GPT-2 Medium) |
| Loss | Cross-entropy trên logits của class | Cross-entropy trên logits của token |
| Kết quả | Spam/Ham | Response bằng ngôn ngữ tự nhiên |
| Đánh giá | Accuracy, F1 | Validation loss, đánh giá response |

**Vì sao giữ LM head?** Vì GPT phải chọn token tiếp theo từ toàn bộ vocabulary, không chỉ chọn một nhãn.

## 4. Instruction dataset

Một mẫu dữ liệu thường có 3 trường:

```python
example = {
    "instruction": "Translate to Vietnamese",
    "input": "I love machine learning",
    "output": "Tôi yêu học máy",
}
```

- `instruction`: Cần thực hiện nhiệm vụ gì.
- `input`: Nội dung bổ sung; có thể rỗng.
- `output`: Câu trả lời mẫu (target).

Chia dataset thành **train / validation / test**; không đánh giá cuối cùng trên chính dữ liệu đã dùng để train.

## 5. Prompt formatting (Alpaca-style)

Chuyển dictionary thành chuỗi văn bản có cấu trúc nhất quán:

```text
Below is an instruction that describes a task.

### Instruction:
Translate to Vietnamese

### Input:
I love machine learning

### Response:
Tôi yêu học máy
```

```python
def format_input(example):
    prompt = (
        "Below is an instruction that describes a task.\n\n"
        f"### Instruction:\n{example['instruction']}"
    )
    if example["input"]:
        prompt += f"\n\n### Input:\n{example['input']}"
    return prompt + "\n\n### Response:\n"

prompt = format_input(example)
training_text = prompt + example["output"]
```

- **Training:** Dùng `prompt + response`.
- **Inference:** Chỉ cung cấp `prompt`, để GPT tự sinh response.

## 6. Tokenization và Dataset

GPT xử lý **token IDs**, không trực tiếp xử lý chuỗi Python.

```python
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(training_text)
print(len(token_ids))
```

Các mẫu dài ngắn khác nhau, nên cần `collate_fn` để gom thành batch có cùng sequence length.

## 7. Custom collate function, padding và masking

**Collate function** nhận nhiều danh sách token IDs, thêm end-of-text token, padding, tạo `inputs` và `targets` lệch nhau 1 token, và đánh dấu padding dư bằng `-100`.

Với GPT-2:

```python
PAD_ID = 50256  # GPT-2 end-of-text token; sách dùng để padding
IGNORE_INDEX = -100
```

Ví dụ minh họa (đã chuẩn hóa độ dài):

```text
Chuỗi A: [10, 20, 30, 40, 50256]
Chuỗi B: [50, 60, 50256, 50256, 50256]
```

Sau khi dịch input/target, các vị trí **padding dư** trong target được thay bằng `-100`, nhưng giữ token kết thúc đầu tiên làm target hợp lệ.

```text
A input : [10, 20, 30, 40]
A target: [20, 30, 40, 50256]

B input : [50, 60, 50256, 50256]
B target: [60, 50256, -100, -100]
```

**`ignore_index=-100`** nghĩa là các vị trí target này không đóng góp vào cross-entropy loss. Đây là cách xử lý padding trong notebook chính của sách.

## 8. Input–target shift và teacher forcing

GPT dự đoán token kế tiếp:

```text
Tokens:  [10, 20, 30, 40, 50]
Inputs:  [10, 20, 30, 40]
Targets: [20, 30, 40, 50]
```

```python
inputs = tokens[:-1]
targets = tokens[1:]
```

**Teacher forcing:** Trong training, model được cung cấp các token đúng ở những vị trí trước để dự đoán token tiếp theo. Trong inference, model dùng các token mà chính nó vừa sinh để tiếp tục.

## 9. Hidden states, logits và tensor shapes

Ví dụ dùng GPT-2 Medium:

- `batch_size = 8`
- `seq_len = 128`
- `emb_dim = 1024`
- `vocab_size = 50257`

| Tensor | Shape | Ý nghĩa |
|---|---|---|
| `input_ids` | `(8, 128)` | IDs của các token |
| `hidden_states` | `(8, 128, 1024)` | Biểu diễn token sau Transformer |
| `logits` | `(8, 128, 50257)` | Điểm cho mỗi token trong vocabulary ở mỗi vị trí |
| `targets` | `(8, 128)` | Token IDs cần dự đoán, hoặc `-100` |

Trong Chapter 6, ta có thể lấy `logits[:, -1, :]` cho **phân loại một nhãn**. Trong Chapter 7, ta tính loss trên **nhiều vị trí token**.

## 10. Token-level cross-entropy

Mô hình dự đoán phân phối xác suất token tiếp theo; cross-entropy phạt khi xác suất của token đúng thấp.

\[
L = -\frac{1}{N}\sum_{t \in \text{valid targets}} \log P(y_t\mid x_{\le t})
\]

`N` là số vị trí target hợp lệ (không bị ignore).

```python
import torch.nn.functional as F

logits = model(input_ids)  # (B, T, V)
loss = F.cross_entropy(
    logits.flatten(0, 1),  # (B*T, V)
    targets.flatten(),     # (B*T,)
    ignore_index=-100,
)
```

**Không cần `softmax()` trước `F.cross_entropy()`**: hàm này nhận logits và tự thực hiện phép tính cần thiết.

### Full-sequence loss và response-only loss

- **Full-sequence loss:** Tính trên prompt và response (trừ padding bị ignore). Đây là cách làm trong notebook chính của Chapter 7.
- **Response-only loss:** Đặt target ở phần prompt thành `-100` để chỉ học dự đoán response. Đây là một biến thể/bài tập mở rộng hữu ích.

## 11. Fine-tuning loop

Ví dụ khung huấn luyện (giả sử `model`, `train_loader`, `device` đã được tạo):

```python
import torch
import torch.nn.functional as F

model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

model.train()
for inputs, targets in train_loader:
    inputs = inputs.to(device)
    targets = targets.to(device)

    optimizer.zero_grad()
    logits = model(inputs)
    loss = F.cross_entropy(
        logits.flatten(0, 1),
        targets.flatten(),
        ignore_index=-100,
    )
    loss.backward()
    optimizer.step()
```

Giải thích:

1. `model.train()` bật training mode.
2. `optimizer.zero_grad()` xóa gradient tích lũy từ bước trước.
3. `model(inputs)` thực hiện forward pass.
4. `cross_entropy` tính token-level loss.
5. `loss.backward()` tính gradient bằng backpropagation.
6. `optimizer.step()` cập nhật trainable weights.

### `requires_grad` và `torch.no_grad()`

- `param.requires_grad=False`: Freeze một parameter.
- `with torch.no_grad():`: Tạm thời không ghi graph gradient cho các phép toán; thường dùng khi validation/inference.
- `model.eval()`: Chuyển hành vi Dropout/BatchNorm sang chế độ đánh giá; **không tự tắt gradient**.

Chapter 7 chủ yếu minh họa instruction fine-tuning của pretrained GPT, không yêu cầu thay classification head như Chapter 6.

## 12. Inference: autoregressive generation

Khi dự đoán, chỉ đưa prompt:

```text
### Instruction:
Explain what a neural network is.

### Response:
```

GPT sinh lần lượt token 1 → token 2 → ... cho đến khi gặp end-of-text hoặc chạm giới hạn token.

```python
model.eval()
with torch.no_grad():
    output_ids = generate(
        model=model,
        idx=prompt_ids,
        max_new_tokens=100,
        context_size=1024,
        eos_id=50256,
    )
```

**Chú ý:** `generate` là hàm được định nghĩa trong code của sách; `prompt_ids` phải là tensor token IDs đúng shape, nằm trên cùng device với model. Sau khi sinh, decode IDs rồi tách phần response khỏi prompt.

## 13. Evaluation

### 13.1 Training/validation loss

Theo dõi loss qua các bước hoặc epoch để kiểm tra quá trình học và dấu hiệu overfitting.

### 13.2 Qualitative evaluation

So sánh câu trả lời mô hình với câu trả lời tham khảo và tự kiểm tra:

- Có làm đúng instruction không?
- Nội dung có chính xác không?
- Có đầy đủ, rõ ràng không?

### 13.3 LLM-as-a-Judge

Sử dụng một LLM khác để chấm chất lượng response theo tiêu chí. Chapter 7 có ví dụ dùng Llama 3 qua Ollama. **Điểm chấm của LLM không tuyệt đối đúng**, nên nên kiểm tra thủ công một số mẫu.

## 14. LoRA (mở rộng, không phải nội dung chính Chapter 7)

**LoRA — Low-Rank Adaptation**: Freeze trọng số gốc, huấn luyện thêm các ma trận nhỏ để thích nghi mô hình với nhiệm vụ mới.

\[
W' = W + BA
\]

- `W`: Trọng số pretrained.
- `A`, `B`: Ma trận adapter được học.
- `r`: Rank nhỏ, giúp giảm số trainable parameters.

LoRA được trình bày kỹ hơn ở **Appendix E** của sách. Nên học sau khi hiểu fine-tuning cơ bản.

## 15. Những lỗi thường gặp

| Lỗi | Nguyên nhân / cách sửa |
|---|---|
| Shape không khớp khi tính loss | Flatten logits thành `(B*T, V)` và targets thành `(B*T,)` |
| Model học padding | Đặt target padding dư thành `-100` và dùng `ignore_index=-100` |
| Tính softmax rồi mới cross-entropy | Đưa logits trực tiếp vào `F.cross_entropy` |
| Validation tốn nhiều memory | Dùng `model.eval()` cùng `torch.no_grad()` |
| Inference không dừng | Kiểm tra `eos_id`, `max_new_tokens`, prompt formatting |
| Loss thấp nhưng response kém | Đánh giá trên prompt chưa thấy và kiểm tra chất lượng dataset |
| Nhầm classification với instruction | Instruction fine-tuning vẫn cần LM head dự đoán vocabulary |

## 16. Checklist phải nắm

- [ ] Phân biệt pretraining, classification fine-tuning và instruction fine-tuning.
- [ ] Hiểu `instruction`, `input`, `output`.
- [ ] Viết được prompt formatting.
- [ ] Biết tokenize và tổ chức dataset.
- [ ] Hiểu `collate_fn`, padding và `ignore_index=-100`.
- [ ] Hiểu input-target shift và teacher forcing.
- [ ] Giải thích được logits shape `(B, T, V)`.
- [ ] Biết vì sao giữ language modeling head.
- [ ] Hiểu token-level cross-entropy và training loop.
- [ ] Phân biệt training và inference.
- [ ] Biết cách đánh giá response.
- [ ] Hiểu LoRA ở mức tổng quan.

## 17. Bài thực hành Chapter 7

**Mục tiêu:** Fine-tune GPT để làm theo các instruction đơn giản.

1. Load instruction dataset.
2. Chia train/validation/test.
3. Format instruction thành prompt-response.
4. Tokenize.
5. Viết custom collate function.
6. Tạo DataLoader.
7. Load pretrained GPT-2 (hoặc bản nhỏ hơn nếu tài nguyên hạn chế).
8. Chạy training loop và theo dõi train/validation loss.
9. Generate response trên các prompt chưa thấy.
10. So sánh trước/sau fine-tuning, ghi lại ví dụ đúng/sai.

**Project tổng hợp Chapters 3–7:** Tự triển khai Attention → GPT Architecture → Pretraining demo → Fine-tune Classification và Fine-tune Instruction thành **hai nhánh riêng** từ pretrained model.

## 18. Tóm tắt một trang

```text
CHAPTER 7 — INSTRUCTION FINE-TUNING

Mục tiêu: GPT học làm theo instruction.
Dataset: instruction + input + output.
Prompt: Format thống nhất, ví dụ Alpaca-style.
Tokenize: Text → token IDs.
Collate: Padding + input-target shift + ignore padding.
Model: Pretrained GPT + giữ LM head.
Logits: (batch, seq_len, vocab_size).
Loss: Token-level cross-entropy.
Train: zero_grad → forward → loss → backward → step.
Infer: Prompt → generate từng token → response.
Eval: Validation loss + chất lượng câu trả lời.
LoRA: Fine-tuning tiết kiệm tham số (mở rộng).
```

## 19. Tài liệu

- Source code chính thức: https://github.com/rasbt/LLMs-from-scratch/tree/main/ch07
- Notebook Chapter 7: https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/ch07.ipynb
- Google Colab: https://colab.research.google.com/github/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/ch07.ipynb

> **Kết luận:** Chapter 7 không tạo một kiến trúc GPT hoàn toàn mới. Ta dùng GPT đã pretrain, chuẩn bị instruction-response data và tiếp tục huấn luyện bằng next-token prediction để GPT học trả lời theo yêu cầu.
